# CH14 — the SYSMON on a two-line display

The top row is the die temperature. The bottom row is whichever reading the
board's potentiometer is pointing at. Both come from the UltraScale+ SYSMON,
read by a **MicroBlaze-V** in the PL, and both reach the glass over the
**AXI IIC** core on the Grove connector.

This notebook is the loader. Nothing on the board runs until it says so.

## What had to change in the hardware, and why

CH14 shipped a design with a MicroBlaze-V, an AXI IIC, a System Management
Wizard — and a Zynq MPSoC whose only job was to supply `pl_clk0` and
`pl_resetn0`. `PSU__USE__M_AXI_GP0/1/2` were all `0`, so **the PS had no AXI
master into the PL at all**. There was no way for a notebook to put a program
into the processor, start it, or read a register to check any of it worked.

`project_sysmon_lcd/build.tcl` sources that original block design and adds
three things:

| | |
|---|---|
| `M_AXI_HPM0_LPD` | the missing path from the PS into the PL |
| `mb_bram_ctrl` | the PS's window onto the processor's memory, where the ELF goes |
| `mb_ctrl` | one GPIO bit, OR-ed into the processor's `Reset` and **nothing else** |

The third one is the subtle one. Driving the `proc_sys_reset`'s `aux_reset_in`
would have been the obvious way to hold the processor — and it would also have
reset the interconnect the PS writes the program *through* and the memory it
writes the program *into*.

Finding a third BRAM port looked impossible: a `blk_mem_gen` has two, and the
data and instruction LMB controllers had both. It is not impossible, because
`lmb_bram_if_cntlr` has `C_NUM_LMB`, and it goes to 8. **One** controller now
serves **both** LMB buses through one port, which frees the other for the PS.
That is what PYNQ's own IOP subsystems do.

The local memory also grew from 16 KB to 64 KB. 16 KB was not a tight budget,
it was a wall: Vitis's `XIic` alone is about 20 KB.

## There is no UART on this design

Not on the MDM, not an `axi_uartlite`. The panel is both the output device and
the thing most likely to be broken, so a fault would otherwise present as a
blank display and nothing else.

The application therefore publishes a small struct — a **mailbox** — into its
own memory, and the PS reads it through `mb_bram_ctrl` while the application
runs. That is the whole debug channel, and the cells below lean on it heavily.

## Finding the pieces

The drivers live in `sw/` and the artifacts in `out/`, and where those are
relative to this notebook depends on how it was deployed. `sw/paths.py`
searches rather than assuming — see the comment at the top of it before
"simplifying" this.

In [ ]:
import os
import sys
import time

# sw/ sits either beside this notebook's project directory or one level above
# it, depending on the deployed layout.
here = os.getcwd()
for candidate in (os.path.join(here, os.pardir, os.pardir, "sw"),
                  os.path.join(here, os.pardir, "sw"),
                  os.path.join(here, "sw")):
    candidate = os.path.normpath(candidate)
    if os.path.isdir(candidate):
        sys.path.insert(0, candidate)
        print("drivers:", candidate)
        break
else:
    raise RuntimeError("could not find sw/ from %s" % here)

import ch14
import mailbox
import paths

BITSTREAM = paths.require_artifact("ch14_sysmon_lcd.bit",
                                   [os.path.join(here, os.pardir, "out")])
ELF = paths.require_artifact("ch14_sysmon_lcd.elf",
                             [os.path.join(here, os.pardir, "out")])
print("bitstream:", BITSTREAM)
print("elf:      ", ELF)

## Download the overlay

`mb_ctrl`'s output register powers up with its reset bit **set**, so the
MicroBlaze-V is held from the moment the bitstream lands.

That is deliberate. An all-zero BRAM is an illegal instruction on RISC-V, and
a processor trap-looping while the ELF is written underneath it is not a
useful starting point. The consequence is worth knowing: **loading the overlay
on its own leaves the panel blank.** This notebook is what starts it.

In [ ]:
session = ch14.Ch14.from_overlay(BITSTREAM)
print("overlay loaded; processor halted:", session.mb.is_halted())

## Read the SYSMON from the PS first

Before involving the application at all. If this works, the bitstream is the
right one and the sensors are converting — which means anything that goes
wrong later is software.

### Why the driver goes looking for its own registers

Vitis ships a driver for this IP, and it computes the DRP window at **`0x400`**
for `system_management_wiz`. CH13 scanned all 8 KB of that window on this IP
revision, on this board, and found it at **`0x1400`**.

The two cannot both be right, and being wrong is invisible: an unmapped read
in this window returns **zero** rather than faulting, which is
indistinguishable from a SYSMON that is present, placed, and silently not
converting. CH13 diagnosed it as exactly that, wrongly, for a day.

So neither `sw/sysmon.py` nor the application's `sysmon.c` assumes. Both try
each candidate and accept one only when the registers there agree with things
known *without* asking the SYSMON: a physically possible temperature, and two
supply rails at the voltages this board is built for.

In [ ]:
drp_base = session.sysmon.locate()
print("DRP window found at 0x%04X" % drp_base)
if drp_base != 0x1400:
    print("  note: not the 0x1400 measured on this hardware -- worth re-reading")

reading = session.sysmon.read_all()
print()
print("temperature   %7.2f C   (min %.2f, max %.2f since power-up)"
      % (reading["temperature"], reading["temperature_min"],
         reading["temperature_max"]))
for name, volts in sorted(reading["supplies"].items()):
    print("%-12s  %7.3f V" % (name, volts))
print("pot           %7.3f V   (raw %d, %.1f%% of full scale)"
      % (reading["pot"]["volts"], reading["pot"]["raw"],
         reading["pot"]["fraction"] * 100))

## Start the application

Halt, load, **verify**, release, wait for the first mailbox.

The verify step reads the memory back over the same AXI path. It should be
exact — the LMB side is idle while the processor is held — and if it is not,
that is worth knowing before anything executes rather than after.

The magic word is blanked before the processor is released. Without that, a
previous run's mailbox still sitting in BRAM would answer the first read and
report a heartbeat for an application that never started.

In [ ]:
state = session.start_application(ELF)
print(mailbox.describe(state))

## Is it actually running?

A loaded application that has stopped and a working one look identical from
outside. The heartbeat is the difference.

In [ ]:
before = session.read_mailbox()
time.sleep(2.0)
after = session.read_mailbox()

beats = (after["heartbeat"] - before["heartbeat"]) & 0xFFFFFFFF
print("%d refreshes in 2 s (the loop runs at 5/s)" % beats)
print("alive:", mailbox.is_alive(before, after))
if after["iic_errors"]:
    print("I2C errors: %d, last status %d -- check the Grove cable"
          % (after["iic_errors"], after["last_iic_status"]))

## Do the panel and the PS agree?

**This is the cell that makes the display trustworthy.** Without it, a panel
showing `42.0 C` proves only that it is showing *something*.

Both sides read the same sensor through the same wizard, moments apart, and
convert it with constants that `sw/test_sysmon.py` asserts are identical — the
application in integers because the MicroBlaze-V has no FPU, this side in
floating point. If they agree, the number on the glass is a measurement.

Reading the SYSMON from both at once is safe: the wizard's bridge arbitrates
DRP reads. The IIC is a different matter — see the last cell.

In [ ]:
state = session.read_mailbox()
from_ps = session.sysmon.temperature_celsius()
from_app = state["temperature_c"]

print("temperature   PS %.3f C   application %.3f C   difference %.3f C"
      % (from_ps, from_app, abs(from_ps - from_app)))

pot_ps = session.sysmon.pot_raw()
print("pot (raw)     PS %-6d      application %-6d      difference %d"
      % (pot_ps, state["vp_vn_raw"], abs(pot_ps - state["vp_vn_raw"])))
print()
print("the panel is showing page %d (%s)" % (state["page"], state["page_name"]))

## Turn the knob

The pot picks the page, and the page is what the bottom row shows. Run this
cell and turn the potentiometer: the page follows.

Five bands with hysteresis at the boundaries. The hysteresis is not a
refinement — bands taken straight off the ADC code mean a pot resting on a
boundary flips the page on converter noise several times a second, which is
unreadable.

In [ ]:
print("turn the pot for 15 seconds...")
print()
print("%-8s %-8s %-12s %s" % ("pot", "%", "page", "bottom row shows"))
last = None
deadline = time.time() + 15
while time.time() < deadline:
    state = session.read_mailbox()
    if state["page"] != last:
        print("%-8d %-8.1f %-12s %s"
              % (state["vp_vn_raw"], state["vp_vn_raw"] / 655.36,
                 state["page"], state["page_name"]))
        last = state["page"]
    time.sleep(0.1)
print()
print("visited every page:", "yes" if last is not None else "no -- pot did not move")

## When nothing appears on the glass

The panel cannot report that the panel is broken. This cell drives it from the
PS instead, which separates *the panel or its wiring is broken* from *the
application is broken* — the distinction that, with no UART in this design,
nothing else can make.

**It halts the processor first, and that is not politeness.** Both masters
reach the same AXI IIC core, and two masters interleaving transactions on one
I2C controller produce nonsense rather than an error. The hardware cannot
prevent it; this is a software rule.

In [ ]:
lcd = session.take_panel()          # halts the processor
lcd.write_row(0, "CH14 panel test")
lcd.write_row(1, "PS drove this")
print("if those two lines are on the panel, the panel and its wiring are fine")

## Hand the panel back

In [ ]:
state = session.start_application(ELF)
print(mailbox.describe(state))

## Stop

The processor keeps running until it is told not to. Reprogramming the PL
underneath it, or loading another overlay, would leave it driving an I2C core
that no longer exists.

In [ ]:
session.stop_application()
print("processor halted:", session.mb.is_halted())